# Tarpuy Analytics ML: segmentación con K-Means
Continuación del análisis exploratorio. Reproduce los resultados de las secciones 9 y 14 del informe.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

df = pd.read_excel("/content/dataset_tarpuy_inscritos_2026.xlsx", sheet_name="Dataset_ML")

## 1. Anonimización y validación de consistencia

In [ ]:
df = df.drop(columns=["DNI"])  # dato personal: no se usa en el modelo

print("Instituciones (sin contar Independiente):", df.loc[df["IE"] != "Independiente", "IE"].nunique())
print("Pagos en efectivo:", (df["Medio_Pago"] == "Efectivo").sum())
print("Inicial 4 años:", (df["Grado"] == "Inicial · 4 años").sum())
real_del = df[(df["Origen"] == "Real") & (df["Modalidad"] == "Institucional")]
print("% reales de delegación con asesor:", round(100 * real_del["Con_Asesor"].mean()))
real = df[df["Origen"] == "Real"]
print("Reales inscritos del 29 al 31/10:", (real["Fecha"] >= "2026-10-29").sum(), "de", len(real))

## 2. Variables del modelo y estandarización

In [ ]:
df["Es_Libre"] = (df["Modalidad"] == "Libre").astype(int)
variables = ["Grado_Num", "Tam_IE", "Con_Asesor", "Dias_Antes_Cierre", "Es_Libre"]
X = StandardScaler().fit_transform(df[variables])

## 3. Selección del número de clústeres

In [ ]:
filas = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=20, random_state=42).fit(X)
    filas.append([k, km.inertia_, silhouette_score(X, km.labels_), davies_bouldin_score(X, km.labels_)])
metricas = pd.DataFrame(filas, columns=["k", "Inercia", "Silueta", "Davies_Bouldin"]).round(3)
display(metricas)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(metricas["k"], metricas["Inercia"], marker="o"); a1.set_title("Método del codo"); a1.set_xlabel("k")
a2.plot(metricas["k"], metricas["Silueta"], marker="o", label="Silueta")
a2.plot(metricas["k"], metricas["Davies_Bouldin"], marker="s", label="Davies-Bouldin")
a2.set_title("Métricas de calidad"); a2.set_xlabel("k"); a2.legend()
plt.tight_layout(); plt.show()

## 4. Modelo final (k = 4) y perfil de los segmentos

In [ ]:
modelo = KMeans(n_clusters=4, n_init=20, random_state=42).fit(X)
df["Segmento"] = modelo.labels_
perfil = df.groupby("Segmento").agg(
    Participantes=("ID", "size"),
    Tam_IE=("Tam_IE", "mean"),
    Con_Asesor=("Con_Asesor", "mean"),
    Dias_mediana=("Dias_Antes_Cierre", "median"),
    Es_Libre=("Es_Libre", "mean"),
    Reales=("Origen", lambda s: (s == "Real").sum()),
).round(2)
display(perfil)
# Nota: el número de cada segmento (0-3) puede variar; el informe los nombra S1-S4 según su perfil.

## 5. Validación con datos reales y comparación de algoritmos

In [ ]:
es_real = df["Origen"] == "Real"
Xr = StandardScaler().fit_transform(df.loc[es_real, variables])
kr = KMeans(n_clusters=4, n_init=20, random_state=42).fit(Xr)
print("Solo reales -> silueta:", round(silhouette_score(Xr, kr.labels_), 3), "| DB:", round(davies_bouldin_score(Xr, kr.labels_), 3),
      "| ARI vs modelo completo:", round(adjusted_rand_score(df.loc[es_real, "Segmento"], kr.labels_), 3))

ward = AgglomerativeClustering(n_clusters=4, linkage="ward").fit(X)
print("Ward -> silueta:", round(silhouette_score(X, ward.labels_), 3), "| DB:", round(davies_bouldin_score(X, ward.labels_), 3),
      "| ARI vs K-Means:", round(adjusted_rand_score(modelo.labels_, ward.labels_), 3))

db = DBSCAN(eps=1.0, min_samples=10).fit(X)
print("DBSCAN -> grupos:", len(set(db.labels_)) - (1 if -1 in db.labels_ else 0), "| ruido:", (db.labels_ == -1).sum())

## 6. Exportar resultados para la API

In [ ]:
import json
salida = {"k": 4, "silueta": round(float(silhouette_score(X, modelo.labels_)), 3),
          "segmentos": perfil.reset_index().to_dict(orient="records"),
          "asignaciones": df[["ID", "Segmento"]].to_dict(orient="records")}
json.dump(salida, open("segmentos.json", "w"), ensure_ascii=False, indent=1)
print("segmentos.json generado")